# 02 · Open a GEC as analysis-ready `xarray`

Umbra's GEC products are geocoded, cloud-optimized GeoTIFFs. That means you
do **not** have to download a multi-gigabyte scene to work with it — you can
stream just the window and resolution you need over HTTP range requests
straight into a georeferenced `xarray.DataArray`.

This notebook needs the `load` extra (xarray + rasterio + numpy):

```bash
pip install "umbra-py[load]"
```

> *Contains Umbra open data, licensed under CC BY 4.0.* `to_xarray` copies the
> license and attribution into the array's `attrs` so it rides along with your
> analysis.

## 1 · Find a scene

Same bounded search as notebook 01 — one GEC acquisition is enough.

In [ ]:
from umbra_py import UmbraCatalog

item = next(
    UmbraCatalog().search(
        start="2024-01-01",
        end="2024-12-31",
        product_types=["GEC"],
        max_per_task=1,
        limit=1,
    )
)
print(item.summary())

## 2 · Stream it into `xarray` (no full download)

`to_xarray` reads band 1 of the GEC and returns a 2D `DataArray` with `y`/`x`
coordinates in the raster's native CRS. `max_size` caps the longest side, so
GDAL pulls a matching overview level — cheap. `db=True` converts linear
amplitude to decibels (the radiometrically-meaningful SAR scale) and masks the
non-positive pixels `log10` can't represent.

In [ ]:
from umbra_py import to_xarray

da = to_xarray(item, max_size=512, db=True)

assert da.ndim == 2
assert set(da.dims) == {"y", "x"}
# License + georeferencing metadata travel in .attrs:
assert "attribution" in da.attrs and "crs" in da.attrs
print(da)

## 3 · Do some analysis

It's a normal `DataArray` now — the whole xarray/numpy toolbox applies. The
values are backscatter in dB; nodata and non-positive pixels are `NaN`, so use
the NaN-aware reducers.

In [ ]:
import numpy as np

valid = da.values[np.isfinite(da.values)]
assert valid.size > 0, "the streamed overview was all nodata"
print(
    f"backscatter dB — mean {valid.mean():.1f}, "
    f"p5 {np.percentile(valid, 5):.1f}, p95 {np.percentile(valid, 95):.1f}"
)

# A quick histogram of the SAR amplitude distribution:
try:
    import matplotlib.pyplot as plt

    da.plot.hist(bins=60)
    plt.title(f"{item.id} — GEC backscatter (dB)")
    plt.show()
except ImportError:
    print("(install matplotlib to see the histogram)")

## 4 · Round-trip the CRS with `rioxarray`

The CRS in `attrs` is a standard WKT/PROJ string, so `rioxarray` (or plain
`rasterio`) can re-attach it for reprojection, clipping, or writing back out.

In [ ]:
try:
    import rioxarray  # noqa: F401  (registers the .rio accessor)

    geo = da.rio.write_crs(da.attrs["crs"])
    print("rioxarray CRS:", geo.rio.crs)
except ImportError:
    print("(install rioxarray to re-attach the CRS as a .rio accessor)")

## Downloading the full asset (optional)

Streaming covers most analysis. When you genuinely need the bytes on disk —
a full-resolution GEC is often multiple gigabytes — `download_item` is
resume-safe:

```python
from umbra_py import download_item

paths = download_item(item, dest_dir="downloads", assets=["GEC"])
# then: rioxarray.open_rasterio(paths[0])
```

*Contains Umbra open data, licensed under CC BY 4.0.*